# 05 · ML feature table (store CA_3)
**Business question:** what inputs would a machine-learning forecast need, and can they be built in SQL?

Builds one row per **item × week** for store CA_3 with the target (`units`) and the features a model like LightGBM would learn from:
lags, rolling averages, price change, event and SNAP days, and week of year.

**Rule for every feature:** it may only use information available **before** the week being forecast. Otherwise the model "sees the answer" (this is called data leakage).

> **Run one notebook at a time.** DuckDB allows only one open connection to `practice.duckdb`. Before opening another notebook, use *Kernel → Shut Down Kernel* in this one.

In [ ]:
import duckdb

# Connect to the local DuckDB file built in 00_setup_and_load
con = duckdb.connect('../data/practice.duckdb')

# Memory settings: needed for the 59M-row sales_daily table on a laptop
con.sql("SET memory_limit = '4GB'")
con.sql("SET preserve_insertion_order = false")
con.sql("SET temp_directory = '../data/tmp'")

## 1. Base table: item × week for CA_3
Aggregates daily rows to weeks and adds weekly counts of event and SNAP days.

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE ml_base_ca3 AS
SELECT s.item_id,
       s.dept_id,
       s.cat_id,
       s.wm_yr_wk,
       SUM(s.units)      AS units,                -- target: weekly units
       MAX(s.sell_price) AS sell_price,           -- price is weekly, so MAX just carries it
       SUM(CASE WHEN c.event_name_1 IS NOT NULL THEN 1 ELSE 0 END) AS event_days,
       SUM(c.snap_CA)    AS snap_days
FROM sales_daily s
INNER JOIN calendar c
        ON s.date = c.date                        -- join at day grain, not week
WHERE s.store_id = 'CA_3'
  AND s.sell_price IS NOT NULL                    -- drop weeks before the item launched
  AND s.wm_yr_wk <> 11617                         -- drop the partial last week
GROUP BY s.item_id, s.dept_id, s.cat_id, s.wm_yr_wk
""")

In [ ]:
# Check: total units match the source with the same filters (no rows lost or duplicated)
con.sql("""
SELECT (SELECT SUM(units) FROM ml_base_ca3) AS units_new_table,
       (SELECT SUM(units) FROM sales_daily
         WHERE store_id = 'CA_3'
           AND sell_price IS NOT NULL
           AND wm_yr_wk <> 11617)            AS units_source
""").df()

,units_new_table,units_source
0,11348323.0,11348323.0


**Check passed:** 11,348,323 units in both, so the aggregation neither lost nor duplicated rows.

## 2. Feature table

In [ ]:
con.sql("""
CREATE OR REPLACE TABLE ml_features_ca3 AS
SELECT item_id,
       dept_id,
       cat_id,
       wm_yr_wk,
       units,
       sell_price,
       event_days,
       snap_days,
       wm_yr_wk % 100 AS week_of_yr,                                              -- seasonality: Walmart week number
       LAG(units, 1)  OVER (PARTITION BY item_id ORDER BY wm_yr_wk) AS units_1wk_pr,   -- last week
       LAG(units, 2)  OVER (PARTITION BY item_id ORDER BY wm_yr_wk) AS units_2wk_pr,   -- two weeks ago
       LAG(units, 52) OVER (PARTITION BY item_id ORDER BY wm_yr_wk) AS units_52wk_pr,  -- same week last year
       -- rolling averages end at the previous week, so the current week is never included
       AVG(units) OVER (PARTITION BY item_id ORDER BY wm_yr_wk
                        ROWS BETWEEN 4 PRECEDING AND 1 PRECEDING)  AS avg_units_pr_4wk,
       AVG(units) OVER (PARTITION BY item_id ORDER BY wm_yr_wk
                        ROWS BETWEEN 13 PRECEDING AND 1 PRECEDING) AS avg_units_pr_13wk,
       -- % price change vs last week (promotions and price increases)
       ROUND((sell_price - LAG(sell_price, 1) OVER (PARTITION BY item_id ORDER BY wm_yr_wk)) * 100
             / LAG(sell_price, 1) OVER (PARTITION BY item_id ORDER BY wm_yr_wk), 1) AS price_change_pct_wow
FROM ml_base_ca3
""")

## 3. Inspect one item: FOODS_3_090

In [ ]:
con.sql("""
SELECT *
FROM ml_features_ca3
WHERE item_id = 'FOODS_3_090'
ORDER BY wm_yr_wk
""").df()

,item_id,dept_id,cat_id,wm_yr_wk,units,sell_price,event_days,snap_days,week_of_yr,units_1wk_pr,units_2wk_pr,units_52wk_pr,avg_units_pr_4wk,avg_units_pr_13wk,price_change_pct_wow
0,FOODS_3_090,FOODS_3,FOODS,11101,970.0,1.25,0.0,4.0,1,NaN,NaN,NaN,NaN,NaN,NaN
1,FOODS_3_090,FOODS_3,FOODS,11102,481.0,1.25,1.0,6.0,2,970.0,NaN,NaN,970.000000,970.000000,0.0
2,FOODS_3_090,FOODS_3,FOODS,11103,0.0,1.25,1.0,0.0,3,481.0,970.0,NaN,725.500000,725.500000,0.0
3,FOODS_3_090,FOODS_3,FOODS,11104,0.0,1.25,1.0,0.0,4,0.0,481.0,NaN,483.666667,483.666667,0.0
4,FOODS_3_090,FOODS_3,FOODS,11105,0.0,1.25,0.0,4.0,5,0.0,0.0,NaN,362.750000,362.750000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
272,FOODS_3_090,FOODS_3,FOODS,11612,856.0,1.60,0.0,0.0,12,782.0,703.0,801.0,785.750000,736.923077,0.0
273,FOODS_3_090,FOODS_3,FOODS,11613,927.0,1.60,0.0,0.0,13,856.0,782.0,973.0,812.750000,759.230769,0.0
274,FOODS_3_090,FOODS_3,FOODS,11614,751.0,1.60,3.0,6.0,14,927.0,856.0,1093.0,817.000000,778.384615,0.0
275,FOODS_3_090,FOODS_3,FOODS,11615,935.0,1.60,1.0,4.0,15,751.0,927.0,774.0,829.000000,785.384615,0.0


**Finding: censored demand in FOODS_3_090**
- In weeks 11103–11105 (Feb 2011), units drop from 970 to **zero for three weeks**, while the price stays at \$1.25.
- A top-selling item does not lose all demand overnight with no price change. These zeros are almost certainly **stockouts**: the store had nothing to sell, so sales understate demand. This is called **censored demand**.
- The same pattern feeds into the lag features (e.g. `units_52wk_pr` = 9 in week 11616) and inflates the item's σ and safety stock in notebook 04.
- **Next step:** build a stockout flag (long zero-sales runs for normally high sellers) so the model and safety-stock calculations can exclude or correct these weeks.

**Known limitation:** M5 has no store inventory data, so stockouts can only be inferred from sales patterns. A production demand-sensing setup would use real on-hand inventory.